# 01 Data Understanding

This notebook explains the dataset grain, scale, and validity checks behind the Airfare Intelligence project. The goal is to decide what the data can legitimately support before modeling anything.

## Business Question

Can Expedia airfare observations be used to estimate fare levels, contextualize whether a fare is low or high, and evaluate a historical BOOK/WAIT decision?

The key validity rule is: a BOOK/WAIT target requires repeated observations of the same or comparable itinerary over time.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

DB_PATH = Path('../data/processed/airfare.duckdb')
con = duckdb.connect(DB_PATH.as_posix(), read_only=True)

## Dataset Grain

Each row in the working subset represents one Expedia fare observation for a specific itinerary (`legId`) on a specific `searchDate`.

This is not just a list of flights. It is a panel-like dataset where many itineraries are observed repeatedly over time.

In [ ]:
con.execute('''
select
    count(*) as row_count,
    count(distinct route) as routes,
    count(distinct legId) as leg_ids,
    min(searchDate) as min_search_date,
    max(searchDate) as max_search_date,
    min(flightDate) as min_flight_date,
    max(flightDate) as max_flight_date,
    min(lead_time_days) as min_lead_time_days,
    max(lead_time_days) as max_lead_time_days
from airfare_clean
''').fetchdf()

## Route Summary

The portfolio model uses two high-volume routes so that the project remains reproducible on a local machine while still using over one million rows.

In [ ]:
con.execute('''
select
    route,
    count(*) as row_count,
    count(distinct legId) as leg_ids,
    count(distinct searchDate) as search_dates,
    count(distinct flightDate) as flight_dates,
    round(avg(totalFare), 2) as avg_total_fare,
    round(median(totalFare), 2) as median_total_fare
from airfare_clean
group by route
order by route
''').fetchdf()

## Repeated Itinerary Validation

This check determines whether temporal analysis and historical BOOK/WAIT labels are possible.

In [ ]:
con.execute('''
with leg_profile as (
    select
        route,
        legId,
        count(distinct searchDate) as search_dates,
        count(distinct totalFare) as fare_values
    from airfare_clean
    group by route, legId
)
select
    route,
    count(*) as leg_ids,
    sum(case when search_dates > 1 then 1 else 0 end) as repeated_across_search_dates,
    sum(case when fare_values > 1 then 1 else 0 end) as multiple_fare_values,
    max(search_dates) as max_search_dates_per_leg,
    round(avg(search_dates), 2) as avg_search_dates_per_leg
from leg_profile
group by route
order by route
''').fetchdf()

## Leakage Watchlist

- `legId`: useful for grouping and temporal labels, but not a prediction feature.
- `baseFare`: too close to `totalFare`, so it leaks target information.
- `fareBasisCode`: likely encodes airline fare rules and may be unavailable to a normal user.
- raw `searchDate` and `flightDate`: can introduce time leakage if used carelessly.

The project uses grouped splits by `legId` to keep repeated observations of the same itinerary out of multiple splits.